In [1]:
import pandas as pd
import re,pickle
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer

In [2]:
df=pd.read_csv('News_Articles_Indian_Express.csv')
df.head(2)

,Unnamed: 0,article_id,headline,desc,date,url,articles,article_type,article_length
0,0,INDEXP20000,"Trainer aircraft crashes in Odisha, 2 killed",The two were taken to a nearby hospital in Kam...,"June 8, 2020 11:06:53 am",https://indianexpress.com/article/india/traine...,A two-seater aircraft crashed in Odisha's Dhen...,short,753
1,1,INDEXP19999,Uttarkhand unlock 1.0: Hotel bookings for mini...,All hotels/ B&B/ Homestay & hospitality servic...,"June 8, 2020 11:03:16 am",https://indianexpress.com/article/india/touris...,Hotels located in non-containment zones can re...,long,2424


In [3]:
df.shape

(20000, 9)

check column value

In [4]:
df['articles']

0        A two-seater aircraft crashed in Odisha's Dhen...
1        Hotels located in non-containment zones can re...
2        Four militants were killed and two security pe...
3        Private offices in Mumbai are set to reopen on...
4        Four days after bureaucrat-turned-politician S...
                               ...                        
19995    The Madhya Pradesh government has served showc...
19996    Three months after Rahul Gandhi quit as party ...
19997    SLAMMING THE state for its decision to deposit...
19998    THE state government on Saturday asked the oil...
19999    Less than a week after the abrogation of Artic...
Name: articles, Length: 20000, dtype: object

In [5]:
df.head(1)['headline'].values

array(['Trainer aircraft crashes in Odisha, 2 killed'], dtype=object)

In [6]:
df.head(1)['desc'].values

array(['The two were taken to a nearby hospital in Kamakhyanagar, where doctors declared them dead, Nayak said.'],
      dtype=object)

In [7]:
df.head(1)['articles'].values

array(["A two-seater aircraft crashed in Odisha's Dhenkanal district on Monday, killing a trainee pilot and her instructor, officials said. The trainer aircraft crashed on the tarmac at the Government Aviation Training Institute (GATI) at Birasala in the district, Additional District Magistrate (ADM), Dhenkanal, B K Nayak, said. The two were taken to a nearby hospital in Kamakhyanagar, where doctors declared them dead, Nayak said. Senior police and district officials are at the spot and a probe into the accident would be conducted. The accident might have occurred due to a technical glitch, officials said. Inspector-in-Charge of Kamakhyanagar Police Station, A Dalua, said the trainer was a man and the identities of the deceased were being ascertained."],
      dtype=object)

check number of aticles based on type

In [8]:
df['article_type'].value_counts()

article_type
long     10248
mid       7381
short     2371
Name: count, dtype: int64

shows all columns

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Unnamed: 0      20000 non-null  int64 
 1   article_id      20000 non-null  object
 2   headline        20000 non-null  object
 3   desc            20000 non-null  object
 4   date            20000 non-null  object
 5   url             20000 non-null  object
 6   articles        20000 non-null  object
 7   article_type    20000 non-null  object
 8   article_length  20000 non-null  int64 
dtypes: int64(2), object(7)
memory usage: 1.4+ MB


select the useful columns

In [10]:
articles= df[['article_id', 'headline', 'desc', 'articles', 'article_type','url']]


In [11]:
articles.head()

,article_id,headline,desc,articles,article_type,url
0,INDEXP20000,"Trainer aircraft crashes in Odisha, 2 killed",The two were taken to a nearby hospital in Kam...,A two-seater aircraft crashed in Odisha's Dhen...,short,https://indianexpress.com/article/india/traine...
1,INDEXP19999,Uttarkhand unlock 1.0: Hotel bookings for mini...,All hotels/ B&B/ Homestay & hospitality servic...,Hotels located in non-containment zones can re...,long,https://indianexpress.com/article/india/touris...
2,INDEXP19998,J-K: Four Hizbul militants killed in Shopian e...,The gunfight comes a day after five militants ...,Four militants were killed and two security pe...,short,https://indianexpress.com/article/india/shopai...
3,INDEXP19997,"Mumbai offices to reopen today, with curbs",Mumbai Traffic Police have prepared for a surg...,Private offices in Mumbai are set to reopen on...,mid,https://indianexpress.com/article/india/corona...
4,INDEXP19996,"PDP, NC, PC call for release of all J&K leaders",Faesal and two PDP leaders—Peer Mansoor and Sa...,Four days after bureaucrat-turned-politician S...,long,https://indianexpress.com/article/india/pdp-nc...


checking missing data

In [12]:
articles.isnull().sum()

article_id      0
headline        0
desc            0
articles        0
article_type    0
url             0
dtype: int64

Checking duplicates

In [13]:
articles.duplicated(subset=['headline','desc']).sum()

np.int64(26)

removing duplicate data

In [14]:
articles = articles.drop_duplicates(subset=['headline', 'articles'])
articles.reset_index(drop=True, inplace=True)


In [15]:
articles.shape

(19973, 6)

In [16]:
articles.head(2)

,article_id,headline,desc,articles,article_type,url
0,INDEXP20000,"Trainer aircraft crashes in Odisha, 2 killed",The two were taken to a nearby hospital in Kam...,A two-seater aircraft crashed in Odisha's Dhen...,short,https://indianexpress.com/article/india/traine...
1,INDEXP19999,Uttarkhand unlock 1.0: Hotel bookings for mini...,All hotels/ B&B/ Homestay & hospitality servic...,Hotels located in non-containment zones can re...,long,https://indianexpress.com/article/india/touris...


creating tags

In [17]:
articles['tags'] = articles['headline'] + " " + articles['desc'] + " " + articles['articles']

In [18]:
articles.head(2)

,article_id,headline,desc,articles,article_type,url,tags
0,INDEXP20000,"Trainer aircraft crashes in Odisha, 2 killed",The two were taken to a nearby hospital in Kam...,A two-seater aircraft crashed in Odisha's Dhen...,short,https://indianexpress.com/article/india/traine...,"Trainer aircraft crashes in Odisha, 2 killed T..."
1,INDEXP19999,Uttarkhand unlock 1.0: Hotel bookings for mini...,All hotels/ B&B/ Homestay & hospitality servic...,Hotels located in non-containment zones can re...,long,https://indianexpress.com/article/india/touris...,Uttarkhand unlock 1.0: Hotel bookings for mini...


In [19]:
articles['tags'][0]

"Trainer aircraft crashes in Odisha, 2 killed The two were taken to a nearby hospital in Kamakhyanagar, where doctors declared them dead, Nayak said. A two-seater aircraft crashed in Odisha's Dhenkanal district on Monday, killing a trainee pilot and her instructor, officials said. The trainer aircraft crashed on the tarmac at the Government Aviation Training Institute (GATI) at Birasala in the district, Additional District Magistrate (ADM), Dhenkanal, B K Nayak, said. The two were taken to a nearby hospital in Kamakhyanagar, where doctors declared them dead, Nayak said. Senior police and district officials are at the spot and a probe into the accident would be conducted. The accident might have occurred due to a technical glitch, officials said. Inspector-in-Charge of Kamakhyanagar Police Station, A Dalua, said the trainer was a man and the identities of the deceased were being ascertained."

creating new dataframe

In [20]:
n_article=articles.drop(columns=['headline','desc','articles'])

In [21]:
n_article.head(2)

,article_id,article_type,url,tags
0,INDEXP20000,short,https://indianexpress.com/article/india/traine...,"Trainer aircraft crashes in Odisha, 2 killed T..."
1,INDEXP19999,long,https://indianexpress.com/article/india/touris...,Uttarkhand unlock 1.0: Hotel bookings for mini...


Cleaning the dataframe

In [22]:
ps = PorterStemmer()

In [23]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    text = " ".join([ps.stem(word) for word in text.split()])  # stemming
    return text

In [24]:
n_article['tags'] = n_article['tags'].apply(clean_text)

In [25]:
n_article['tags'][0]

'trainer aircraft crash in odisha kill the two were taken to a nearbi hospit in kamakhyanagar where doctor declar them dead nayak said a twoseat aircraft crash in odisha dhenkan district on monday kill a traine pilot and her instructor offici said the trainer aircraft crash on the tarmac at the govern aviat train institut gati at birasala in the district addit district magistr adm dhenkan b k nayak said the two were taken to a nearbi hospit in kamakhyanagar where doctor declar them dead nayak said senior polic and district offici are at the spot and a probe into the accid would be conduct the accid might have occur due to a technic glitch offici said inspectorincharg of kamakhyanagar polic station a dalua said the trainer wa a man and the ident of the deceas were be ascertain'

creating vector

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=9000,stop_words='english')

In [27]:
vector = cv.fit_transform(n_article['tags'])

Creating Similarity of every vectors with every vectors

In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
similarity = cosine_similarity(vector)

In [30]:
def recommend_from_text(user_text, top_n=5):
  
    user_text = clean_text(user_text)
    
    # Convert input text into vector using SAME vectorizer
    text_vector = cv.transform([user_text]).toarray()
    
    # Compute similarity with all articles vector and text_vector
    similarities = cosine_similarity(text_vector, vector).flatten()
    
    #sort articles by similarities
    article_list=sorted(list(enumerate(similarities)),reverse=True,key=lambda x:x[1])[0:5]

    #Print recommendations
    for i in article_list:
        article_row = articles.iloc[i[0]]
        print(f"Headline: {article_row['headline']}")
        print(f"URL: {article_row['url']}")
        print("-" * 20) 


In [31]:
recommend_from_text("Trainer aircraft crashes in karnataka")


Headline: IAF MiG trainer aircraft crashes near Gwalior airbase, both pilots safe
URL: https://indianexpress.com/article/india/iaf-mig-trainer-aircraft-crashes-near-gwalior-airbase-pilot-eject-safely-6026894/
--------------------
Headline: Navy’s MiG-29K jet crashes near Goa village, pilots eject to safety
URL: https://indianexpress.com/article/india/indian-navy-mig-29-aircraft-crashes-in-goa-live-updates-pilots-eject-to-safety-6122582/
--------------------
Headline: Navy’s MiG 29k aircraft crashes near Goa, pilot ejects safely
URL: https://indianexpress.com/article/india/indian-navy-mig-29k-aircraft-on-routine-mission-crashes-near-goa-pilot-ejects-safely-6282349/
--------------------
Headline: Trainer aircraft crashes in Odisha, 2 killed
URL: https://indianexpress.com/article/india/trainer-aircraft-crashes-in-odisha-2-killed-6448250/
--------------------
Headline: Flight instructor, trainee pilot killed in crash in Telangana
URL: https://indianexpress.com/article/india/instructor-trai

Saving the dataframe, vectorizer, and the vectorized matrix

In [32]:
pickle.dump(articles, open('articles.pkl', 'wb'))
pickle.dump(cv, open('cv.pkl', 'wb'))
pickle.dump(vector, open('vector.pkl', 'wb'))